##
**Imports**

Install hydesign if needed.
Import basic libraries. 
Import HPP model assembly class.
Import the examples file path.

In [ ]:
# Install hydesign if needed
import importlib
if not importlib.util.find_spec("hydesign"):
    !pip install git+https://gitlab.windenergy.dtu.dk/TOPFARM/hydesign.git

In [ ]:
import os
import time
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hydesign.assembly.hpp_assembly import hpp_model
from hydesign.examples import examples_filepath

##
**Specifying the site**

Hydesign, provides example data from several sites in India and Europe. 

The site coordinates (longitude, latitude, and altitude) are given in `examples_sites.csv`.

In [ ]:
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
#examples_sites

select a site to run

In [ ]:
name = 'France_good_wind'
ex_site = examples_sites.loc[examples_sites.name == name]

longitude = ex_site['longitude'].values[0]
latitude = ex_site['latitude'].values[0]
altitude = ex_site['altitude'].values[0]

##

**Input data: weather and electricity price**

For each site a csv input file is provided. This file contains the weather and the spot market electricity prices.

The columns required are:

* Wind speed at multiple heights (`WS_hh`, where hh is the height in meters). In order to estimate the PV cell temperature, it is important to have a wind speed close to the ground (`WS_1`). 

* (optional) Wind direction at multiple heights (`WD_hh`, where hh is the height in meters). Wind directions can be used when the layout of the wind farm is known and therefore the wind generation depends on the wind direction.  

* Air temperature in Kelvin close to the ground (`temp_air_1`)

* Global horizontal irradiance (`ghi`).

* Direct normal irradiance (`dni`).

* Diffuse horizontal irradiance (`dhi`).

* Spot market electricity prices (`Price`)


In [ ]:
input_ts_fn = examples_filepath+ex_site['input_ts_fn'].values[0]

input_ts = pd.read_csv(input_ts_fn, index_col=0, parse_dates=True)

required_cols = [col for col in input_ts.columns if 'WD' not in col]
input_ts = input_ts.loc[:,required_cols]
#input_ts

##
**Additional data**

Each region has a different hybrid power plant evaluation parameter file (`hpp_pars.yml`).
This file contains all the assumptions on the valuation of a hyrbid power plant. 

In the examples, the Indian sites have specified a requiredment to provide power at peak hours. This requirement is specified by setting the minimum number of full power hours at peak prices times per day. 

For example the Indian case penalty has expected hours of `0.85*3= 2.55`:


`n_full_power_hours_expected_per_day_at_peak_price: 2.55`


While the Europe sites have no penalty.

`n_full_power_hours_expected_per_day_at_peak_price: 0`

In [ ]:
sim_pars_fn = examples_filepath+ex_site['sim_pars_fn'].values[0]

with open(sim_pars_fn) as file:
    sim_pars = yaml.load(file, Loader=yaml.FullLoader)

print(sim_pars_fn)    
sim_pars

##
**Wind Turbine parameters:**

1. Hub height [m] 
2. Rotor diameters [m]
3. Rated power of the wind turbine [MW] (`p_rated`)

**PV parameters:**

4. Surface tilt [deg] (`surface_tilt`)
5. Surface azimuth [deg] (`surface_azimuth`)
6. DC-AC ratio [-] (`solar_DCAC`): This parameter controls how much over-planting of PV (in DC power) is connected to the inverters. It is common practice in PV design to have `solar_DCAC = 1.5`. 

In [ ]:
rotor_diameter_m = 220
hub_height_m = 130
wt_rated_power_MW = 10
surface_tilt_deg = 35
surface_azimuth_deg = 180
DC_AC_ratio = 1.5


## 
**Initializing the HPP model**

Initialize the HPP model (hpp_model class) with the coordinates and the necessary input files.

In [ ]:
hpp = hpp_model(
        latitude=latitude,
        longitude=longitude,
        altitude=altitude,
        rotor_diameter_m = rotor_diameter_m,
        hub_height_m = hub_height_m,
        wt_rated_power_MW = wt_rated_power_MW,
        surface_tilt_deg = surface_tilt_deg,
        surface_azimuth_deg = surface_azimuth_deg,
        DC_AC_ratio = DC_AC_ratio,
        num_batteries = 5,
        work_dir = './',
        sim_pars_fn = sim_pars_fn,
        input_ts_fn = input_ts_fn,
)

##
### Evaluating the HPP model

In [ ]:
start = time.time()

Nwt = 20
wind_MW_per_km2 = 7
solar_MW = 150
b_P = 20
b_E_h  = 3
cost_of_batt_degr = 5
clearance = hub_height_m - rotor_diameter_m / 2
sp = 4 * wt_rated_power_MW * 10 ** 6 / np.pi / rotor_diameter_m ** 2

x = [# Wind plant design
    clearance, sp, wt_rated_power_MW, Nwt, wind_MW_per_km2,
    # PV plant design
    solar_MW,  surface_tilt_deg, surface_azimuth_deg, DC_AC_ratio,
    # Energy storage & EMS price constrains
    b_P, b_E_h, cost_of_batt_degr]

outs = hpp.evaluate(*x)

hpp.print_design(x, outs)

end = time.time()
print(f'exec. time [min]:', (end - start)/60 )

In [ ]:
# Metric names
names =  [
    "NPV_over_CAPEX",               # 0
    "NPV_MEuro",                    # 1
    "IRR",                          # 2
    "LCOE_Euro_per_MWh",            # 3
    "Revenues_MEuro",               # 4
    "CAPEX_MEuro",                  # 5
    "OPEX_MEuro",                  
    "Wind_CAPEX_MEuro",             
    "Wind_OPEX_MEuro",              
    "PV_CAPEX_MEuro",               
    "PV_OPEX_MEuro",                
    "Battery_CAPEX_MEuro",          
    "Battery_OPEX_MEuro",           
    "Shared_CAPEX_MEuro",           
    "Shared_OPEX_MEuro",            
    "Penalty_lifetime_MEuro",       
    "Mean_Annual_Electricity_Sold_GWh", 
    "GUF",                          
    "Grid_MW",                       
    "Wind_MW",                        
    "Solar_MW",                       
    "Battery_Energy_MWh",            
    "Battery_Power_MW",              
    "Total_curtailment_GWh",         
    "Total_curtailment_with_deg_GWh",
    "Awpp_km2",                       
    "Apvp_km2",                        
    "Plant_area_km2",                  
    "Rotor_diam_m",                    
    "Hub_height_m",                    
    "Number_of_batteries_lifetime",    
    "Break_even_PPA_Euro_per_MWh",     
    "Capacity_factor_wind",            
    "AEP_GWh",                         
    "AEP_with_degradation_GWh"         # 34
]

# Original values
original = np.array([7.13853491e-01, 2.42052848e+02, 1.14633056e-01, 3.56192867e+01,
       3.32123652e+01, 3.39079168e+02, 5.64742893e+00, 2.22901416e+02,
       4.63492893e+00, 5.02500000e+01, 1.01250000e+00, 6.37432351e+00,
       0.00000000e+00, 5.95534286e+01, 0.00000000e+00, 0.00000000e+00,
       8.73667959e+02, 3.32445951e-01, 3.00000000e+02, 2.00000000e+02,
       1.50000000e+02, 6.00000000e+01, 2.00000000e+01, 2.78578965e+01,
       5.05908426e+00, 2.85714286e+01, 1.83900000e+00, 2.85714286e+01,
       2.20000000e+02, 1.30000000e+02, 2.00000000e+00, 2.38069835e+01,
       3.91147482e-01, 1.00553023e+03, 8.74237174e+02])
# exec. time: 0.155 min
##
# values with ems_core cplex
# original = np.array([7.09379723e-01, 2.40046894e+02, 1.14347578e-01, 3.55542602e+01,
#        3.30786929e+01, 3.38389844e+02, 5.64742893e+00, 2.22901416e+02,
#        4.63492893e+00, 5.02500000e+01, 1.01250000e+00, 5.68500000e+00,
#        0.00000000e+00, 5.95534286e+01, 0.00000000e+00, 0.00000000e+00,
#        8.73822949e+02, 3.32504927e-01, 3.00000000e+02, 2.00000000e+02,
#        1.50000000e+02, 6.00000000e+01, 2.00000000e+01, 1.33717903e+01,
#        5.97435310e+00, 2.85714286e+01, 1.83900000e+00, 2.85714286e+01,
#        2.20000000e+02, 1.30000000e+02, 1.00000000e+00, 2.37675101e+01,
#        3.91147482e-01, 1.00553023e+03, 8.74237174e+02])

# Values with ems_core pyomo_cplex
# original = np.array([7.19836038e-01, 2.40907415e+02, 1.15064102e-01, 3.52270171e+01,
#        3.29565916e+01, 3.34669844e+02, 5.64742893e+00, 2.22901416e+02,
#        4.63492893e+00, 5.02500000e+01, 1.01250000e+00, 1.96500000e+00,
#        0.00000000e+00, 5.95534286e+01, 0.00000000e+00, 0.00000000e+00,
#        8.73963800e+02, 3.32558524e-01, 3.00000000e+02, 2.00000000e+02,
#        1.50000000e+02, 6.00000000e+01, 2.00000000e+01, 0.00000000e+00,
#        6.83432686e+00, 2.85714286e+01, 1.83900000e+00, 2.85714286e+01,
#        2.20000000e+02, 1.30000000e+02, 0.00000000e+00, 2.35734766e+01,
#        3.91147482e-01, 1.00553023e+03, 8.74237174e+02])

# Compute differences
diff = outs - original
rel_diff = np.where(original != 0, diff / original, np.nan)
rel_pct = rel_diff * 100.0

# Print header
print(f"{'Metric':35} {'Original':>12} {'New':>12} {'Diff':>12} {'Rel%':>10}")
print("-"*85)

# Print row by row
for i in range(len(names)):
    print(f"{names[i]:35} {original[i]:12.6f} {outs[i]:12.6f} {diff[i]:12.6f} {rel_pct[i]:10.5f}")


In [ ]:
import pickle
with open(r"C:\code\hydesign\hydesign\tests\test_files\ems_input_ems.pickle", "rb") as f:
    input_ems = pickle.load(f)

input_ems